# R²D-HOPE-MoRE — Training on Google Colab

**What this notebook does:**
1. Mounts Google Drive (checkpoints survive session restarts)
2. Installs dependencies
3. Clones / uploads the `r2d_hope` package
4. Trains + evaluates a BPE tokenizer on WikiText-103 (free, no API keys)
5. Pre-trains R²D-HOPE-MoRE with a cosine LR schedule and gradient accumulation
6. Logs to TensorBoard (live in Colab)
7. Saves checkpoints to Drive every 1 000 steps
8. Optionally applies SA++ block sparsification after warmup

**Runtime:** Use `Runtime → Change runtime type → T4 GPU` (free tier).
A100 40GB (Colab Pro) will be ~5× faster.

**Expected throughput on T4:**
- `d_model=384`, `seq_len=256`, `batch=8×accum4=32` → ~18 000 tok/s
- 20 000 steps × 32 × 256 = ~163M tokens total (solid LM pre-training signal)
- Wall-clock: ~2.5 h on T4

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 0 — Mount Google Drive

In [13]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/r2d_hope_training'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/r2d_hope_training


## Step 1 — Install Dependencies

In [14]:
%%capture install_output
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets tokenizers accelerate
!pip install -q tensorboard
print('Installation done.')

## Step 2 — Upload or Clone the r2d_hope Package

**Option A (recommended):** Clone from GitHub if you've pushed the repo.
```
!git clone https://github.com/YOUR_USERNAME/R2D-HOPE-VLM.git /content/R2D-HOPE-VLM
%cd /content/R2D-HOPE-VLM
```

**Option B:** Upload the `r2d_hope/` folder as a zip directly from your machine.
Run the cell below to use the file-upload widget.

In [15]:
import os, sys

# ── CHOOSE ONE OPTION ──────────────────────────────────────────────────────
SETUP_MODE = 'github'   # 'github' | 'upload' | 'drive'
GITHUB_URL = 'https://github.com/TracNg99/R2D-HOPE-MoRE.git'  # change me
DRIVE_REPO_ZIP = '/content/drive/MyDrive/r2d_hope.zip'             # for 'drive'
# ──────────────────────────────────────────────────────────────────────────

REPO_DIR = '/content/R2D-HOPE-VLM'

if SETUP_MODE == 'github':
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {GITHUB_URL} {REPO_DIR}')
    else:
        os.system(f'git -C {REPO_DIR} pull')

elif SETUP_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()          # pick r2d_hope.zip
    zip_name = list(uploaded.keys())[0]
    os.makedirs(REPO_DIR, exist_ok=True)
    os.system(f'unzip -q {zip_name} -d {REPO_DIR}')

elif SETUP_MODE == 'drive':
    os.makedirs(REPO_DIR, exist_ok=True)
    os.system(f'unzip -q {DRIVE_REPO_ZIP} -d {REPO_DIR}')

# Add to path so `import r2d_hope` works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Smoke test
from r2d_hope import R2DConfig, R2D_HOPE_MoRE
print('r2d_hope package loaded successfully.')

r2d_hope package loaded successfully.


## Step 3 — GPU Check

In [16]:
import torch
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    
    gpu = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:      {gpu}  ({mem_gb:.1f} GB VRAM)')
    # Recommend dtype
    if 'A100' in gpu or 'H100' in gpu:
        RECOMMENDED_DTYPE = 'bfloat16'
    elif 'T4' in gpu or 'V100' in gpu:
        RECOMMENDED_DTYPE = 'float16'
    else:
        RECOMMENDED_DTYPE = 'float32'
    print(f'Recommended dtype: {RECOMMENDED_DTYPE}')

PyTorch:  2.10.0+cu128
CUDA:     True
GPU:      Tesla T4  (15.6 GB VRAM)
Recommended dtype: float16


## Step 4 — Configuration

Two configs provided:
- **`cfg_small`** — 6.5M params, fits in 4 GB VRAM, fast iteration on T4
- **`cfg_medium`** — 11.5M params, needs ~8 GB VRAM, best results on T4 Pro / A100

Change `MODEL_SIZE` below.

In [17]:
from r2d_hope import R2DConfig, TrainConfig

MODEL_SIZE = 'small'   # 'small' | 'medium'

# ── Model configs ──────────────────────────────────────────────────────────
if MODEL_SIZE == 'small':
    model_cfg = R2DConfig(
        d_model=384,
        d_ffn=1024,
        d_embedding=192,
        num_heads=6,
        head_dim=64,
        vocab_size=16384,
        nested_depth=20,
        num_experts=4,
        top_k_experts=2,
        window_size=128,
        ddim_inference_steps=10,
    )
else:  # medium
    model_cfg = R2DConfig(
        d_model=512,
        d_ffn=1280,
        d_embedding=256,
        num_heads=8,
        head_dim=64,
        vocab_size=16384,
        nested_depth=20,
        num_experts=4,
        top_k_experts=2,
        window_size=128,
        ddim_inference_steps=10,
    )

# ── Training config ────────────────────────────────────────────────────────
# Adjust batch_size / seq_len if you hit OOM:
#   T4  (15 GB): small model → batch=8,  seq=512  is safe
#   T4  (15 GB): medium model → batch=4, seq=256 to be safe
#   A100 (40GB): medium model → batch=16, seq=512 works fine
train_cfg = TrainConfig(
    run_name         = f'r2d_hope_{MODEL_SIZE}',
    checkpoint_dir   = f'{DRIVE_ROOT}/checkpoints',
    log_dir          = f'{DRIVE_ROOT}/logs',
    tokenizer_dir    = f'{DRIVE_ROOT}/tokenizer',
    dataset_name     = 'wikitext',
    dataset_config   = 'wikitext-103-raw-v1',
    seq_len          = 512,
    batch_size       = 8 if MODEL_SIZE == 'small' else 4,
    grad_accum_steps = 4,
    lr               = 3e-4,
    lr_min           = 3e-5,
    warmup_steps     = 500,
    max_steps        = 20_000,
    eval_every       = 500,
    save_every       = 1_000,
    log_every        = 50,
    dtype            = RECOMMENDED_DTYPE,
    vocab_size       = model_cfg.vocab_size,
    use_sapp         = False,   # set True to enable SA++ pruning after warmup
    sapp_warmup_steps= 8_000,
)

print(f'Model config: {MODEL_SIZE}')
print(f'  d_model={model_cfg.d_model}, vocab={model_cfg.vocab_size}, '
      f'nested_depth={model_cfg.nested_depth}')
print(f'Train config: seq_len={train_cfg.seq_len}, '
      f'effective_batch={train_cfg.batch_size * train_cfg.grad_accum_steps}, '
      f'max_steps={train_cfg.max_steps}, dtype={train_cfg.dtype}')

Model config: small
  d_model=384, vocab=16384, nested_depth=20
Train config: seq_len=512, effective_batch=32, max_steps=20000, dtype=float16


## Step 5 — Tokenizer

Trains a 16k BPE tokenizer on WikiText-103 (~50 MB of English text) and caches it to Drive.
Subsequent runs load it instantly.

In [18]:
from r2d_hope import build_or_load_tokenizer

tokenizer = build_or_load_tokenizer(
    tokenizer_dir   = train_cfg.tokenizer_dir,
    vocab_size      = train_cfg.vocab_size,
    dataset_name    = 'wikitext',
    dataset_config  = 'wikitext-103-raw-v1',
    max_train_chars = 50_000_000,
)

# Sanity check
test_enc = tokenizer('The quick brown fox jumps', return_tensors='pt')
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Test encode: {test_enc["input_ids"].tolist()}')
print(f'Test decode: {tokenizer.decode(test_enc["input_ids"][0])}')

Tokenizer ready — vocab size: 16384
Vocab size: 16384
Test encode: [[55, 206, 2366, 4373, 10393, 554, 9724]]
Test decode: T he Ġquick Ġbrown Ġfox Ġj umps


## Step 6 — Build Model and Count Parameters

In [19]:
import torch
from r2d_hope import R2D_HOPE_MoRE
model = R2D_HOPE_MoRE(model_cfg)
param_info = model.count_parameters()
total_params = param_info["total"]
trainable = param_info["trainable"]
print(f'Total parameters:     {total_params:,}  ({total_params/1e6:.2f}M)')
print(f'Trainable parameters: {trainable:,}  ({trainable/1e6:.2f}M)')
# Quick forward pass to confirm no errors
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = getattr(torch, train_cfg.dtype)
model  = model.to(device)
with torch.no_grad(), torch.autocast(device_type=device, dtype=dtype):
    dummy_prompt = torch.randint(1, model_cfg.vocab_size, (2, 4)).to(device)
    dummy_answer = torch.randint(1, model_cfg.vocab_size, (2, 8)).to(device)
    dummy_ctx    = torch.zeros(2, 1, model_cfg.d_model, device=device, dtype=dtype)
    out = model(dummy_prompt, dummy_answer, dummy_ctx)
print(f'Forward pass OK — loss={out["loss"].item():.4f}')


Total parameters:     13,651,585  (13.65M)
Trainable parameters: 13,651,585  (13.65M)
Forward pass OK — loss=2.1209


## Step 7 — Data Loaders

In [20]:
from r2d_hope import make_pretrain_loader

train_loader = make_pretrain_loader(
    tokenizer,
    seq_len        = train_cfg.seq_len,
    batch_size     = train_cfg.batch_size,
    dataset_name   = train_cfg.dataset_name,
    dataset_config = train_cfg.dataset_config,
    split          = "train",
)

# Eval on WikiText-103 validation split
eval_loader = make_pretrain_loader(
    tokenizer,
    seq_len        = train_cfg.seq_len,
    batch_size     = train_cfg.batch_size,
    dataset_name   = train_cfg.dataset_name,
    dataset_config = train_cfg.dataset_config,
    split          = "validation",
)

# Peek at one batch
import itertools
sample_batch = next(itertools.islice(iter(train_loader), 1))
print(f'Batch input_ids: {sample_batch["input_ids"].shape}')
print(f'Batch labels:    {sample_batch["labels"].shape}')
print(f'Sample tokens:   {sample_batch["input_ids"][0, :16].tolist()}')
print(f'Sample text:     {tokenizer.decode(sample_batch["input_ids"][0, :32])}')

TypeError: make_pretrain_loader() got an unexpected keyword argument 'split'

## Step 8 — TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {train_cfg.log_dir}

## Step 9 — (Optional) SA++ Block Sparsification Setup

Skip this cell if `use_sapp = False` in your `train_cfg`.
If enabled, the pruner wraps the model's target Linear layers and adds
group-L1 regularisation during training. After `sapp_warmup_steps`, the
trainer automatically prunes to structured sparsity.

In [ ]:
pruner = None

if train_cfg.use_sapp:
    from r2d_hope import SABlockPruner
    pruner = SABlockPruner(
        sparsity_lambda = train_cfg.sapp_lambda,
        block_size      = (16, 16),
    )
    pruner.wrap(model)
    n_masks = len(pruner._masks)
    extra_params = sum(m.log_importance.numel() for _, m in pruner._masks)
    print(f'SA++ enabled: {n_masks} layers wrapped, {extra_params:,} extra importance params')
    print(f'Pruning will trigger at step {train_cfg.sapp_warmup_steps}')
else:
    print('SA++ disabled — training dense model.')

## Step 10 — Train 🚀

Training resumes automatically from the latest Drive checkpoint if one exists.
If the Colab session dies, just re-run from Step 0 — it picks up where it left off.

In [ ]:
from r2d_hope import train

history = train(
    model        = model,
    train_loader = train_loader,
    eval_loader  = eval_loader,
    cfg          = train_cfg,
    pruner       = pruner,
)

print('\nTraining complete!')
if history['eval_ppl']:
    best_step, best_ppl = min(history['eval_ppl'], key=lambda x: x[1])
    print(f'Best eval perplexity: {best_ppl:.2f} at step {best_step}')

## Step 11 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
if history['step'] and history['train_loss']:
    axes[0].plot(history['step'], history['train_loss'], lw=1.5)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

# Perplexity curve
if history['eval_ppl']:
    eval_steps = [s for s, _ in history['eval_ppl']]
    eval_ppls  = [p for _, p in history['eval_ppl']]
    axes[1].plot(eval_steps, eval_ppls, marker='o', lw=1.5, color='darkorange')
    axes[1].set_title('Eval Perplexity')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('PPL')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/training_curves.png', dpi=150)
plt.show()
print(f'Saved to {DRIVE_ROOT}/training_curves.png')

## Step 12 — Inference Test

In [ ]:
import torch

model.eval()
device = next(model.parameters()).device
dtype  = getattr(torch, train_cfg.dtype)

PROMPT = 'The history of artificial intelligence'
NUM_TOKENS = 32

prompt_ids = tokenizer.encode(PROMPT, return_tensors='pt').to(device)
ctx        = torch.zeros(1, 1, model_cfg.d_model, device=device, dtype=dtype)

with torch.no_grad(), torch.autocast(device_type=device.type, dtype=dtype):
    generated = model.generate(
        prompt_ids, ctx,
        num_answer_tokens = NUM_TOKENS,
        ddim_steps        = model_cfg.ddim_inference_steps,
    )

output_text = tokenizer.decode(generated[0], skip_special_tokens=True)
print(f'Prompt:  {PROMPT}')
print(f'Output:  {output_text}')

## Step 13 — (Optional) Prune with SA++ and Re-evaluate

Run this only if you trained with `use_sapp = False` and want to apply
post-hoc pruning to a converged model.

In [ ]:
from r2d_hope import SABlockPruner
from r2d_hope.trainer import evaluate

# Evaluate before pruning
pre_metrics = evaluate(model, eval_loader, device, dtype, max_batches=100)
print(f'Before pruning — PPL: {pre_metrics["eval_ppl"]:.2f}')

# Prune
posthoc_pruner = SABlockPruner(block_size=(16, 16))
posthoc_pruner.wrap(model)
posthoc_pruner.prune(model)
print(posthoc_pruner.report())

# Evaluate after pruning
post_metrics = evaluate(model, eval_loader, device, dtype, max_batches=100)
print(f'After pruning  — PPL: {post_metrics["eval_ppl"]:.2f}')
print(f'PPL degradation: +{post_metrics["eval_ppl"] - pre_metrics["eval_ppl"]:.2f}')

## Step 14 — Save Final Model to Drive

In [ ]:
import torch, os
final_path = os.path.join(DRIVE_ROOT, f'final_{MODEL_SIZE}.pt')
torch.save({
    'model_state': model.state_dict(),
    'model_cfg':   vars(model_cfg),
    'train_cfg':   vars(train_cfg),
}, final_path)
print(f'Final model saved to {final_path}')